# 06 — Más Allá del Curso: Paradigmas Avanzados
**Algoritmos y Estructuras de Datos · Universidad de Talca**

---
> *"Este notebook es un mapa, no un destino. Muestra las rutas que existen más allá de lo que hemos recorrido."*

| Campo | Detalle |
|---|---|
| **Tópico** | S03 — Diseño de Algoritmos |
| **Notebook** | 06 de 06 — **CIERRE DEL BLOQUE** |
| **Duración estimada** | 45 minutos |
| **Prerequisito** | NB00–NB05 (panorama del bloque S03) |
| **Objetivo** | Motivación panorámica — NO profundizar, sino despertar curiosidad |

---
### ¿Qué veremos?

| Sección | Paradigma | Pregunta central |
|---|---|---|
| A | **Algoritmos de Aproximación** | ¿Y si el problema es NP-difícil? |
| B | **Búsqueda Local / Metaheurísticas** | ¿Y si no podemos explorar todo el espacio? |
| C | **Algoritmos Aleatorizados** | ¿Y si le damos un dado al algoritmo? |
| D | **¿Qué sigue?** | ¿Adónde llevan estos caminos? |

In [ ]:
# ── Verificación de dependencias ────────────────────────────────────────────
import sys
# Todo el notebook usa la librería estándar de Python (visualizaciones en texto y
# pruebas con unittest); ipywidgets solo se usa en los widgets interactivos.
_reqs = {'ipywidgets': 'ipywidgets'}
_missing = []
for pkg, mod in _reqs.items():
    try:
        __import__(mod)
        print(f'✓ {pkg}')
    except ImportError:
        _missing.append(pkg)
        print(f'✗ {pkg} — falta')
if _missing:
    print(f'\nInstalar: !pip install {" ".join(_missing)}')
else:
    print(f'\nPython {sys.version.split()[0]} — listo.')

In [ ]:
from IPython.display import display
import ipywidgets as widgets
from ipywidgets import Output, IntSlider, FloatSlider, HBox, VBox
import random, math
from itertools import combinations
from typing import List, Tuple, Optional, Dict

# ── Color del texto en los widgets HTML ─────────────────────────────────────
TEXTO    = '#212121'

print('✓ Librerías cargadas')

---
## Sección A — Algoritmos de Aproximación

### A.1 El problema: NP-difícil

Todos los paradigmas del curso (Greedy, DP, BT, B&B) buscan la solución **óptima**.  
Pero existe una clase de problemas —los **NP-difíciles**— para los que no se conoce ningún algoritmo eficiente que garantice el óptimo.

**Ejemplos de problemas NP-difíciles:**
- Viajante de Comercio (TSP): visitar n ciudades con distancia mínima
- Coloración de grafos: colorear un grafo con k colores
- Cobertura de vértices (Vertex Cover)
- Problema de satisfacibilidad (SAT)

Para estos problemas, la estrategia es diferente:

> **¿Qué pasa si en lugar del óptimo buscamos una solución que sea como máximo *α* veces el óptimo?**

Eso es un **algoritmo de aproximación con razón α**.

### A.2 Ejemplo: 2-Aproximación para Cobertura de Vértices

**Problema:** Dado un grafo G=(V,E), encuentra el subconjunto mínimo de vértices que "cubre" todas las aristas (cada arista tiene al menos un extremo en el conjunto).

**Algoritmo 2-aprox** (greedy, O(V+E)):
```
cobertura = {}
para cada arista (u,v) no cubierta:
    agregar u y v a la cobertura
    marcar todas las aristas de u y v como cubiertas
```

**¿Por qué da a lo más 2× el óptimo?**  
Cada arista que tomamos es "independiente" (ningún vértice de esta arista apareció antes). El óptimo debe incluir al menos uno de {u,v} para cubrir esa arista. Nosotros tomamos los dos → a lo más el doble.

In [ ]:
# ── 2-Aproximación para Vertex Cover ────────────────────────────────────────

def vertex_cover_2aprox(
        grafo: Dict[int, List[int]],
        verbose: bool = True
) -> List[int]:
    '''
    Algoritmo 2-aproximación para Minimum Vertex Cover.

    Garantía: |cobertura| <= 2 × |cobertura_óptima|

    Complejidad:
        Tiempo: O(V + E)
        Espacio: O(V)
    '''
    cobertura = set()

    for u in grafo:
        for v in grafo[u]:
            if u in cobertura or v in cobertura:
                continue        # la arista ya está cubierta: no se agrega nada
            # Arista libre: el óptimo necesita al menos uno de sus extremos; tomamos ambos
            cobertura.add(u)
            cobertura.add(v)
            if verbose:
                print(f'  Arista ({u},{v}) libre → agregar ambos. '
                      f'Cobertura: {sorted(cobertura)}')

    return sorted(cobertura)


def vertex_cover_optimo(grafo: Dict[int, List[int]]) -> List[int]:
    '''
    Cobertura mínima exacta por fuerza bruta: prueba subconjuntos de menor a mayor tamaño.

    Solo sirve para grafos pequeños: es la búsqueda exponencial que la
    aproximación evita.

    Complejidad:
        Tiempo: O(2^V · E)
        Espacio: O(V + E)
    '''
    aristas = {(min(u, v), max(u, v)) for u in grafo for v in grafo[u]}
    for k in range(len(grafo) + 1):
        for candidato in combinations(grafo, k):
            elegidos = set(candidato)
            if all(u in elegidos or v in elegidos for u, v in aristas):
                return sorted(elegidos)
    return sorted(grafo)


# Grafo de ejemplo: 8 nodos
GRAFO_VC = {
    0: [1, 3],
    1: [0, 2, 4],
    2: [1, 5],
    3: [0, 4, 6],
    4: [1, 3, 5, 7],
    5: [2, 4],
    6: [3, 7],
    7: [4, 6],
}

print('Grafo de ejemplo (8 nodos):')
print('Ejecutando 2-aproximación para Vertex Cover...')
print()
cobertura_vc = vertex_cover_2aprox(GRAFO_VC)
cobertura_opt = vertex_cover_optimo(GRAFO_VC)
razon = len(cobertura_vc) / len(cobertura_opt)
print(f'\nCobertura 2-aprox : {cobertura_vc}')
print(f'Tamaño            : {len(cobertura_vc)}')
print(f'Óptimo real       : {cobertura_opt} (fuerza bruta) → {len(cobertura_opt)} nodos')
print(f"Razón             : {razon:.2f}x ≤ 2.0x  {'✓' if razon <= 2 else '✗'}")

In [ ]:
# ── Visualización en texto: grafo con la cobertura marcada ──────────────────
# GRAFO_VC es una grilla (0-1-2 / 3-4-5 / 6-7) y se dibuja con sus nodos en posición.
# [n] = nodo en la cobertura, (n) = nodo fuera de la cobertura.

def marca(nodo: int, cobertura: List[int]) -> str:
    '''Retorna "[n]" si el nodo está en la cobertura y "(n)" si no.'''
    return f'[{nodo}]' if nodo in cobertura else f'({nodo})'


def dibujar_grafo_vc(cobertura: List[int]) -> None:
    '''Imprime la grilla de GRAFO_VC marcando los nodos de la cobertura.'''
    m = [marca(nodo, cobertura) for nodo in range(len(GRAFO_VC))]
    print(f' {m[0]}───{m[1]}───{m[2]}')
    print('  │     │     │')
    print(f' {m[3]}───{m[4]}───{m[5]}')
    print('  │     │')
    print(f' {m[6]}───{m[7]}')


print('Grafo original:\n')
dibujar_grafo_vc([])

print('\nCobertura 2-aprox ([n] = en la cobertura, (n) = fuera):\n')
dibujar_grafo_vc(cobertura_vc)

print('\nCobertura óptima (fuerza bruta):\n')
dibujar_grafo_vc(cobertura_opt)

aristas_vc = sorted({(min(u, v), max(u, v)) for u in GRAFO_VC for v in GRAFO_VC[u]})
print(f'\nAristas ({len(aristas_vc)}) y extremos que las cubren:')
for u, v in aristas_vc:
    extremos = [x for x in (u, v) if x in cobertura_vc]
    estado = 'cubierta por ' + ' y '.join(map(str, extremos)) if extremos else '✗ sin cubrir'
    print(f'  {u} ── {v}   {estado}')

### A.3 ¿Qué significa ser 2-aproximado?

```
OPT = tamaño del óptimo real (desconocido, calcularlo sería NP-difícil)
ALG = tamaño de nuestra solución

Garantía: ALG ≤ 2 × OPT
```

En el ejemplo: OPT = 4 (la fuerza bruta encuentra {0, 2, 4, 6}), ALG = 8 → 8 ≤ 2×4 = 8 ✓

Esta grilla alcanza justo el peor caso de la garantía: cada arista libre que toma el
algoritmo aporta sus dos extremos, y el óptimo solo necesita uno de cada una.

**Conexión con lo visto en el curso:**  
> Es como Greedy, pero con una **garantía matemática** de qué tan lejos podemos estar del óptimo.

| Paradigma | Garantía de optimalidad |
|---|---|
| Greedy | Ninguna en general (puede fallar) |
| DP / BT / B&B | 100% óptimo (pero puede ser exponencial) |
| **Aproximación** | **A lo más α × OPT en tiempo polinomial** |

---
## Sección B — Búsqueda Local y Metaheurísticas

### B.1 El problema: espacios enormes con muchos óptimos locales

Imagina optimizar una función con **cientos de máximos locales**. Backtracking exploraría todo el espacio, lo que es impracticable. Las metaheurísticas exploran de forma inteligente sin garantizar el óptimo, pero funcionan muy bien en la práctica.

### B.2 Hill Climbing

```
solución = inicio_aleatorio()
mientras haya mejora:
    vecino = mejor_vecino(solución)
    si f(vecino) > f(solución):
        solución = vecino
    sino:
        parar  ← ¡atrapado en óptimo local!
```

**Problema:** se queda atascado en el primer óptimo local que encuentra.

### B.3 Simulated Annealing (Recocido Simulado)

Inspirado en el recocido del metal: al enfriar metal lentamente, los átomos encuentran una configuración de energía mínima.

```
T = temperatura_inicial   ← controla la aleatoriedad
solución = inicio_aleatorio()
mientras T > T_final:
    vecino = vecino_aleatorio(solución)
    Δ = f(vecino) - f(solución)
    si Δ > 0:                           ← mejora: siempre acepta
        solución = vecino
    sino con prob. exp(Δ/T):            ← empeora: acepta con probabilidad
        solución = vecino
    T = T × α                           ← enfriamiento gradual
```

La clave: al principio (T alto) acepta soluciones peores → escapa de óptimos locales.  
Al final (T bajo) se vuelve más selectivo → converge a una buena solución.

In [ ]:
# ── Función multimodal para demostración ────────────────────────────────────

def f_multimodal(x: float) -> float:
    '''Función con varios máximos locales y un máximo global.'''
    return (math.sin(x) + math.sin(2.7 * x) + math.sin(0.5 * x)
            + 0.1 * math.sin(10 * x))


def hill_climbing(
        f, x_init: float, paso: float = 0.05,
        x_min: float = 0.0, x_max: float = 10.0,
        max_iter: int = 1000
) -> Tuple[float, float, List[Tuple[float, float]]]:
    '''
    Hill Climbing: busca máximo local moviendo +/- paso.

    Complejidad:
        Tiempo: O(max_iter)
        Espacio: O(max_iter) historial
    '''
    x = x_init
    historial = [(x, f(x))]
    for _ in range(max_iter):
        candidatos = []
        for delta in [-paso, paso]:
            nx = x + delta
            if x_min <= nx <= x_max:
                candidatos.append((f(nx), nx))
        if not candidatos:
            break
        mejor_f, mejor_x = max(candidatos)
        if mejor_f <= f(x):
            break
        x = mejor_x
        historial.append((x, f(x)))
    return x, f(x), historial


def simulated_annealing(
        f, x_init: float, T0: float = 5.0, T_fin: float = 0.01,
        alpha: float = 0.995, paso: float = 0.5,
        x_min: float = 0.0, x_max: float = 10.0,
        seed: int = 42
) -> Tuple[float, float, List[Tuple[float, float]]]:
    '''
    Simulated Annealing: acepta soluciones peores con probabilidad exp(Δ/T).

    Complejidad:
        Tiempo: O(log(T0/T_fin) / log(1/alpha)) iteraciones
        Espacio: O(iteraciones) historial
    '''
    rng = random.Random(seed)
    x = x_init
    mejor_x = x
    mejor_f = f(x)
    T = T0
    historial = [(x, f(x))]
    while T > T_fin:
        delta_x = rng.uniform(-paso, paso)
        nx = max(x_min, min(x_max, x + delta_x))
        delta_f = f(nx) - f(x)
        if delta_f > 0:
            x = nx
        elif rng.random() < math.exp(delta_f / T):
            x = nx
        if f(x) > mejor_f:
            mejor_f = f(x)
            mejor_x = x
        historial.append((x, f(x)))
        T *= alpha
    return mejor_x, mejor_f, historial


# Probar con inicio en el mismo punto
X_INIT = 0.5
x_hc, f_hc, hist_hc = hill_climbing(f_multimodal, X_INIT)
x_sa, f_sa, hist_sa = simulated_annealing(f_multimodal, X_INIT)

# Óptimo real por búsqueda densa
xs_grid = [10 * i / 9999 for i in range(10000)]   # 10 000 puntos equiespaciados en [0, 10]
x_opt_real = max(xs_grid, key=f_multimodal)
f_opt_real = f_multimodal(x_opt_real)

print(f'Inicio         : x={X_INIT:.2f}, f={f_multimodal(X_INIT):.4f}')
print(f'Hill Climbing  : x={x_hc:.4f}, f={f_hc:.4f}  (pasos: {len(hist_hc)})')
print(f'Simul. Anneal. : x={x_sa:.4f}, f={f_sa:.4f}  (pasos: {len(hist_sa)})')
print(f'Óptimo real    : x={x_opt_real:.4f}, f={f_opt_real:.4f}')
print()
gap_hc = (f_opt_real - f_hc) / abs(f_opt_real) * 100
gap_sa = (f_opt_real - f_sa) / abs(f_opt_real) * 100
print(f'Gap HC  vs óptimo: {gap_hc:.2f}%')
print(f'Gap SA  vs óptimo: {gap_sa:.2f}%')

In [ ]:
# ── Visualización en texto: Hill Climbing vs Simulated Annealing ─────────────

def grafico_funcion(f, marcas: Dict[str, float], x_min: float = 0.0,
                    x_max: float = 10.0, ancho: int = 60, alto: int = 15) -> None:
    '''
    Dibuja f en una grilla de caracteres: '·' para la curva y una letra por punto marcado.

    Parámetros:
        f: función de una variable.
        marcas (dict): {letra: x}. Si dos marcas caen en la misma celda, se ve la última.
        x_min, x_max (float): intervalo que se grafica.
        ancho, alto (int): columnas y filas de la grilla.

    Complejidad:
        Tiempo: O(ancho · alto)
        Espacio: O(ancho · alto)
    '''
    xs = [x_min + (x_max - x_min) * c / (ancho - 1) for c in range(ancho)]
    ys = [f(x) for x in xs]
    todos = ys + [f(x) for x in marcas.values()]
    y_min, y_max = min(todos), max(todos)

    def fila(y: float) -> int:
        return round((y_max - y) / (y_max - y_min) * (alto - 1))

    grilla = [[' '] * ancho for _ in range(alto)]
    for c, y in enumerate(ys):
        grilla[fila(y)][c] = '·'
    for letra, x in marcas.items():
        c = round((x - x_min) / (x_max - x_min) * (ancho - 1))
        grilla[fila(f(x))][c] = letra

    for r, caracteres in enumerate(grilla):
        y = y_max - (y_max - y_min) * r / (alto - 1)
        print(f'{y:6.2f} │' + ''.join(caracteres))
    print(' ' * 7 + '└' + '─' * ancho)
    etiqueta_max = f'{x_max:g}'
    print(' ' * 8 + f'{x_min:g}'.ljust(ancho - len(etiqueta_max)) + etiqueta_max)


print('f(x) en [0, 10] — * = óptimo real, I = inicio, H = Hill Climbing, S = Simulated Annealing\n')
grafico_funcion(f_multimodal, {'*': x_opt_real, 'I': X_INIT, 'H': x_hc, 'S': x_sa})

print(f'\nHill Climbing: {len(hist_hc)} pasos, x = {X_INIT:.2f} → {x_hc:.3f}, f = {f_hc:.4f} '
      '(se detiene en el primer máximo local)')

# Traza de SA: estado cada 10 % de las iteraciones (T₀ = 5.0 y α = 0.995, los valores por defecto)
print('\nSimulated Annealing — estado cada 10 % de las iteraciones')
print(f"{'Avance':>6}  {'Iter':>5}  {'T':>6}  {'x':>6}  {'f(x)':>7}  {'Mejor f':>7}")
print('-' * 46)
ultima = len(hist_sa) - 1
for pct in range(0, 101, 10):
    k = pct * ultima // 100
    x_k, f_k = hist_sa[k]
    mejor_k = max(fh for _, fh in hist_sa[:k + 1])
    print(f'{pct:>5}%  {k:>5}  {5.0 * 0.995 ** k:>6.3f}  {x_k:>6.3f}  {f_k:>7.4f}  {mejor_k:>7.4f}')

In [ ]:
# ── Widget interactivo: SA con temperatura configurable ─────────────────────

T0_sl    = FloatSlider(value=5.0, min=0.1, max=20.0, step=0.5,
                       description='T inicial:', style={'description_width': '90px'})
alpha_sl = FloatSlider(value=0.995, min=0.90, max=0.999, step=0.001,
                       description='Enfriamiento α:', style={'description_width': '110px'},
                       readout_format='.3f')
seed_sl  = IntSlider(value=42, min=0, max=100, step=1,
                     description='Semilla:', style={'description_width': '80px'})
out_meta = Output()

def _actualizar_meta(change=None):
    with out_meta:
        out_meta.clear_output(wait=True)
        T0     = T0_sl.value
        alpha  = alpha_sl.value
        seed   = seed_sl.value
        x_s, f_s, hist_s = simulated_annealing(
            f_multimodal, X_INIT, T0=T0, alpha=alpha, seed=seed)

        print('f(x) — * = óptimo real, S = mejor punto encontrado por SA\n')
        grafico_funcion(f_multimodal, {'*': x_opt_real, 'S': x_s})

        # Enfriamiento: temperatura cada 10 % de las iteraciones, en barras de texto
        print(f'\nEnfriamiento (T₀={T0}, α={alpha:.3f})')
        ultima = len(hist_s) - 1
        for pct in range(0, 101, 10):
            k = pct * ultima // 100
            T_k = T0 * alpha ** k
            print(f"  iter {k:>5} │{'█' * round(T_k / T0 * 40):<40} T={T_k:.3f}")

        gap = (f_opt_real - f_s) / abs(f_opt_real) * 100
        print(f'\nSA encontró f={f_s:.4f} (óptimo={f_opt_real:.4f}, gap={gap:.2f}%)')
        print(f'Iteraciones: {len(hist_s)}')


T0_sl.observe(_actualizar_meta, names='value')
alpha_sl.observe(_actualizar_meta, names='value')
seed_sl.observe(_actualizar_meta, names='value')
display(VBox([
    widgets.HTML('<h3 style="color:#212121;margin-bottom:4px">'
                 'Simulated Annealing — Configuración</h3>'),
    HBox([T0_sl, alpha_sl, seed_sl]),
    out_meta
]))
_actualizar_meta()

### B.4 Conexión con lo visto en el curso

> Simulated Annealing es como **Backtracking aleatorizado**: en lugar de explorar sistemáticamente todo el árbol, salta por el espacio de soluciones con cierta dosis de aleatoriedad controlada.

| Paradigma | Garantía | Velocidad | Úsalo cuando |
|---|---|---|---|
| Backtracking | Óptimo garantizado | Exponencial | n pequeño, necesitas exactitud |
| Hill Climbing | Solo óptimo local | O(iter) | Función bien comportada, sin muchos óptimos locales |
| **Simulated Annealing** | **Buena solución en práctica** | **O(iter)** | **n grande, calidad práctica > exactitud** |

**Aplicaciones reales de SA:**
- Diseño de chips VLSI (colocar millones de transistores)
- Planificación de rutas de aviones
- Alineamiento de secuencias genéticas

---
## Sección C — Algoritmos Aleatorizados

### C.1 Dos categorías

| Categoría | Tiempo | Resultado | Ejemplo |
|---|---|---|---|
| **Las Vegas** | Aleatorio | **Siempre correcto** | QuickSort con pivot aleatorio |
| **Monte Carlo** | Fijo | Correcto con alta probabilidad | Estimación de π |

### C.2 Las Vegas — QuickSort aleatorizado

QuickSort determinístico: si el pivot siempre cae en el extremo → O(n²).
QuickSort aleatorizado: pivot al azar → O(n log n) **esperado** en cualquier entrada.

**Conexión con D&V:** es Divide y Vencerás, pero el punto de división es aleatorio.  
El resultado es **siempre correcto** (la lista siempre queda ordenada). Solo el tiempo varía.

In [ ]:
# ── QuickSort aleatorizado (Las Vegas) ──────────────────────────────────────

def quicksort_det(arr: List[int]) -> Tuple[List[int], int]:
    '''
    QuickSort determinístico (pivot = último elemento).
    Retorna (lista_ordenada, comparaciones).

    Complejidad:
        Tiempo: O(n²) peor caso (entrada ordenada), O(n log n) promedio
        Espacio: O(n) recursión
    '''
    comps = [0]

    def _qs(a: List[int]) -> List[int]:
        if len(a) <= 1:
            return a
        pivot = a[-1]    # siempre el último
        izq, der = [], []
        for x in a[:-1]:
            comps[0] += 1
            (izq if x <= pivot else der).append(x)
        return _qs(izq) + [pivot] + _qs(der)

    return _qs(arr[:]), comps[0]


def quicksort_rand(arr: List[int], seed: int = None) -> Tuple[List[int], int]:
    '''
    QuickSort aleatorizado (pivot uniformemente al azar).
    Retorna (lista_ordenada, comparaciones).

    Complejidad:
        Tiempo: O(n log n) esperado en CUALQUIER entrada
        Espacio: O(n) recursión
    '''
    rng = random.Random(seed)
    comps = [0]

    def _qs(a: List[int]) -> List[int]:
        if len(a) <= 1:
            return a
        idx = rng.randint(0, len(a) - 1)
        pivot = a[idx]
        resto = a[:idx] + a[idx+1:]
        izq, der = [], []
        for x in resto:
            comps[0] += 1
            (izq if x <= pivot else der).append(x)
        return _qs(izq) + [pivot] + _qs(der)

    return _qs(arr[:]), comps[0]


# Peor caso para QS determinístico: arreglo ya ordenado
n_test = 200
arr_peor = list(range(n_test))          # ordenado = peor para det
arr_rand = list(range(n_test))
random.Random(7).shuffle(arr_rand)      # aleatorio

_, comp_det_peor = quicksort_det(arr_peor)
_, comp_rand_peor = quicksort_rand(arr_peor, seed=42)
_, comp_det_norm = quicksort_det(arr_rand)
_, comp_rand_norm = quicksort_rand(arr_rand, seed=42)

print(f'n = {n_test}')
print(f'{"Entrada":<25} {"QS Det":>10} {"QS Rand":>10}')
print('-' * 47)
print(f'{"Ordenado (peor caso)":<25} {comp_det_peor:>10,} {comp_rand_peor:>10,}')
print(f'{"Aleatorio (caso normal)":<25} {comp_det_norm:>10,} {comp_rand_norm:>10,}')
print()
ref_nlogn = int(n_test * math.log2(n_test))
ref_n2    = n_test ** 2
print(f'Referencia n log n ≈ {ref_nlogn:,}')
print(f'Referencia n²      = {ref_n2:,}')
print()
print('→ QS aleatorizado evita el peor caso O(n²) sin importar la entrada.')

In [ ]:
# ── Monte Carlo — Estimación de π ───────────────────────────────────────────

def estimar_pi_montecarlo(
        n_puntos: int, seed: int = 42
) -> Tuple[float, List[float]]:
    '''
    Estima π usando el método de Monte Carlo.

    Idea: puntos aleatorios en cuadrado [0,1]×[0,1].
    Proporción dentro del círculo unitario ≈ π/4.

    Complejidad:
        Tiempo: O(n)
        Espacio: O(n) historial
    Error esperado: O(1/√n) — baja muy lentamente
    '''
    rng = random.Random(seed)
    dentro = 0
    estimaciones = []
    for i in range(1, n_puntos + 1):
        x = rng.random()
        y = rng.random()
        if x*x + y*y <= 1.0:
            dentro += 1
        estimaciones.append(4 * dentro / i)
    return estimaciones[-1], estimaciones


N_MC = 5000
pi_est, hist_pi = estimar_pi_montecarlo(N_MC)

# Puntos en el cuadrado [0,1]×[0,1] dibujados en una grilla de 40×20 caracteres
# (cada carácter es el doble de alto que de ancho, así la grilla se ve cuadrada).
# • = dentro del círculo, × = fuera. El borde entre ambos dibuja el arco de radio 1.
rng_vis = random.Random(42)
N_VIS = 500
xs_mc = [rng_vis.random() for _ in range(N_VIS)]
ys_mc = [rng_vis.random() for _ in range(N_VIS)]

ANCHO_MC, ALTO_MC = 40, 20
grilla_mc = [[' '] * ANCHO_MC for _ in range(ALTO_MC)]
dentro_vis = 0
for x, y in zip(xs_mc, ys_mc):
    dentro = x**2 + y**2 <= 1
    dentro_vis += dentro
    col = min(int(x * ANCHO_MC), ANCHO_MC - 1)
    fila = min(int((1 - y) * ALTO_MC), ALTO_MC - 1)
    grilla_mc[fila][col] = '•' if dentro else '×'
pi_aprox_vis = 4 * dentro_vis / N_VIS

print(f'Monte Carlo — n={N_VIS} puntos: π ≈ {pi_aprox_vis:.4f}   (• dentro, × fuera)\n')
print('  1 ┌' + '─' * ANCHO_MC + '┐')
for caracteres in grilla_mc:
    print('    │' + ''.join(caracteres) + '│')
print('  0 └' + '─' * ANCHO_MC + '┘')
print('    0' + ' ' * (ANCHO_MC - 1) + '1')

# Convergencia: estimación acumulada tras distintas cantidades de puntos
print(f'\nConvergencia ({N_MC} muestras) → π ≈ {pi_est:.6f}   (π real = {math.pi:.6f})\n')
print('┌────────┬──────────┬─────────┐')
print('│ puntos │ π ≈      │ error % │')
print('├────────┼──────────┼─────────┤')
for n_i in [10, 50, 100, 500, 1000, 2000, N_MC]:
    est = hist_pi[n_i - 1]
    print(f'│ {n_i:>6} │ {est:<8.5f} │ {abs(est - math.pi) / math.pi * 100:>7.3f} │')
print('└────────┴──────────┴─────────┘')

error_pct = abs(pi_est - math.pi) / math.pi * 100
print(f'Error con n={N_MC}: {error_pct:.4f}%')
print(f'Para reducir el error a la mitad necesitaríamos n={N_MC * 4} puntos (error ∝ 1/√n).')

In [ ]:
# ── Comparación: QS det vs QS rand — comparaciones por tamaño ─────────────

ns_qs = [50, 100, 200, 400, 800]
comps_det_peor_l, comps_rand_peor_l = [], []
comps_det_rand_l, comps_rand_rand_l = [], []

for n_i in ns_qs:
    a_peor = list(range(n_i))
    a_al   = list(range(n_i))
    random.Random(7).shuffle(a_al)
    _, cd_p  = quicksort_det(a_peor)
    _, cr_p  = quicksort_rand(a_peor, seed=42)
    _, cd_al = quicksort_det(a_al)
    _, cr_al = quicksort_rand(a_al, seed=42)
    comps_det_peor_l.append(cd_p)
    comps_rand_peor_l.append(cr_p)
    comps_det_rand_l.append(cd_al)
    comps_rand_rand_l.append(cr_al)

# Tabla con las cuatro combinaciones y las curvas de referencia
print('Comparaciones de QuickSort por tamaño y tipo de entrada\n')
print(f"{'n':>4} │ {'Det ordenado':>12} │ {'Rand ordenado':>13} │ {'Det aleatorio':>13} │ "
      f"{'Rand aleatorio':>14} │ {'n log n':>7} │ {'n²':>7}")
print('─' * 5 + '┼' + '─' * 14 + '┼' + '─' * 15 + '┼' + '─' * 15 + '┼' + '─' * 16 + '┼' + '─' * 9 + '┼' + '─' * 9)
for k, n_i in enumerate(ns_qs):
    print(f'{n_i:>4} │ {comps_det_peor_l[k]:>12,} │ {comps_rand_peor_l[k]:>13,} │ '
          f'{comps_det_rand_l[k]:>13,} │ {comps_rand_rand_l[k]:>14,} │ '
          f'{int(n_i * math.log2(n_i)):>7,} │ {n_i**2:>7,}')

def mostrar_barras(filas, unidad="ms", ancho=40, formato="8.2f", maximo=None):
    """
    Imprime un gráfico de barras horizontal en texto.

    Parámetros:
        filas (list[tuple[str, float]]): pares (etiqueta, valor).
        unidad (str): unidad que se muestra junto a cada valor.
        ancho (int): caracteres de la barra que representa a `maximo`.
        formato (str): formato del valor impreso, por ejemplo "8.2f" o "8,.0f".
        maximo (float | None): valor que ocupa todo el ancho; por defecto, el mayor
            de `filas`. Pasar el mismo valor a dos gráficos los deja en la misma escala.
    """
    maximo = maximo or max(valor for _, valor in filas) or 1
    margen = max(len(etiqueta) for etiqueta, _ in filas)
    for etiqueta, valor in filas:
        barra = "█" * round(valor / maximo * ancho) or ("▏" if valor > 0 else "")
        print(f"{etiqueta:>{margen}} │{barra:<{ancho}} {valor:{formato}} {unidad}")


# Barras para el n más grande: el peor caso determinístico domina a todo el resto
n_max = ns_qs[-1]
print(f'\nComparaciones con n = {n_max}\n')
mostrar_barras([
    ('QS Det — entrada ordenada', comps_det_peor_l[-1]),
    ('QS Rand — entrada ordenada', comps_rand_peor_l[-1]),
    ('QS Det — entrada aleatoria', comps_det_rand_l[-1]),
    ('QS Rand — entrada aleatoria', comps_rand_rand_l[-1]),
    ('n log n (referencia)', int(n_max * math.log2(n_max))),
], unidad='comparaciones', formato='8,.0f')


### C.3 Las Vegas vs Monte Carlo — Resumen

| | Las Vegas | Monte Carlo |
|---|---|---|
| **Resultado** | Siempre correcto | Correcto con prob. p |
| **Tiempo** | Aleatorio (esperado acotado) | Determinístico |
| **Cuándo falla** | Tarda mucho (rara vez) | Da resultado incorrecto (con prob. pequeña) |
| **Ejemplo** | QuickSort aleatorizado | Estimación de π, tests de primalidad |
| **Reducir error** | Ejecutar de nuevo si tarda | Aumentar n (error ∝ 1/√n) |

> **Conexión con el curso:** QuickSort aleatorizado es Divide y Vencerás con punto de división aleatorio. La aleatorización *rompe la adversarialidad*: ninguna entrada puede ser siempre el peor caso.

---
## Sección D — ¿Qué sigue?

### D.1 Panorama completo de paradigmas

| Paradigma | Idea central | Garantía | Complejidad típica | Dónde aprender más |
|---|---|---|---|---|
| Divide y Vencerás | Dividir → Vencer → Combinar | Óptimo | O(n log n) | CLRS Cap. 4, Kleinberg Cap. 5 |
| Algoritmos Voraces | Mejor decisión local en cada paso | Solo si se puede probar | O(n log n) | CLRS Cap. 15, Kleinberg Cap. 4 |
| Programación Dinámica | Tabla de subproblemas solapados | Óptimo | O(n·W) o O(n²) | CLRS Cap. 14, AtCoder DP Contest |
| Backtracking | Búsqueda exhaustiva con retroceso | Óptimo | O(2ⁿ) o O(n!) | Skiena Cap. 9 |
| Branch & Bound | BT + cota superior/inferior | Óptimo | O(2ⁿ) práct. menor | Skiena Cap. 9 |
| **Aprox.** | Solución α-óptima en tiempo polinomial | α × OPT | Polinomial | CLRS Cap. 35 |
| **Metaheurísticas** | Búsqueda aleatoria inteligente | Buena solución práctica | O(iter) | Nocedal & Wright |
| **Aleatorizados** | Aleatorizar para romper casos adversariales | Correcto (LV) / prob. (MC) | O(n log n) esp. | MitzenmacherUpfal |
| **Prog. Lineal Entera** | Optimización continua/discreta | Óptimo (si LP factible) | Exp. peor, polinomial práct. | Cormen Apéndice, Bertsimas |

### D.2 Cursos donde profundizar

| Curso (típico) | Paradigmas que profundiza |
|---|---|
| **Diseño y Análisis de Algoritmos** (postgrado) | Aproximación, Aleatorizados, Complejidad NP |
| **Investigación de Operaciones** | Programación Lineal Entera, B&B avanzado |
| **Inteligencia Artificial** | Metaheurísticas, Búsqueda heurística (A*), RL |
| **Machine Learning** | Optimización estocástica (SGD, Adam) |
| **Computación de Alto Rendimiento** | Algoritmos paralelos y distribuidos |

In [ ]:
# ── Mapa de conexiones entre paradigmas (estilo comando `tree`) ─────────────
# Cada nodo es (nombre, relación con su padre, visto en este curso, hijos).
MAPA = ('DISEÑO DE ALGORITMOS', '', True, [
    ('Divide y Vencerás', '', True, [
        ('Algoritmos Aleatorizados', 'aleatoriza', False, []),
    ]),
    ('Greedy', '', True, [
        ('Aproximación', 'garantía α×OPT', False, []),
    ]),
    ('Programación Dinámica', '', True, []),
    ('Backtracking', '', True, [
        ('Branch & Bound', 'añade cota', True, [
            ('Programación Lineal Entera', 'continuo', False, []),
        ]),
        ('Metaheurísticas', 'aleatoriza', False, []),
    ]),
])


def mostrar_mapa(nodo: tuple, prefijo: str = '', conector: str = '') -> None:
    '''
    Imprime el mapa de paradigmas como el comando `tree`.

    Los paradigmas de cursos siguientes se marcan con ⇢ y la relación con
    su padre va entre paréntesis.

    Complejidad:
        Tiempo: O(nodos)
        Espacio: O(profundidad) por la recursión
    '''
    nombre, relacion, en_curso, hijos = nodo
    texto = nombre if en_curso else f'⇢ {nombre}'
    if relacion:
        texto += f'   ({relacion})'
    print(prefijo + conector + texto)
    sangria = prefijo + {'': '', '├── ': '│   ', '└── ': '    '}[conector]
    for k, hijo in enumerate(hijos):
        mostrar_mapa(hijo, sangria, '└── ' if k == len(hijos) - 1 else '├── ')


print('Mapa de Paradigmas Algorítmicos — S03 Diseño de Algoritmos')
print('(sin marca = en este curso · ⇢ = siguientes cursos)\n')
mostrar_mapa(MAPA)

In [ ]:
# ── Resumen interactivo: ¿Qué paradigma usar? ────────────────────────────────

GUIA = [
    {
        'pregunta': '¿El problema se puede dividir en subproblemas del mismo tipo?',
        'si_idx': 1, 'no_texto': '→ Considera Greedy o fuerza bruta'
    },
    {
        'pregunta': '¿Los subproblemas se repiten (se solapan)?',
        'si_idx': 2, 'no_texto': '→ Divide y Vencerás (NB01)'
    },
    {
        'pregunta': '¿La solución local óptima garantiza el óptimo global?',
        'si_texto': '→ Greedy (NB02)',
        'no_texto': '→ Programación Dinámica (NB03)'
    },
]

GUIA2 = [
    ('¿Necesito explorar todo el espacio?', [
        ('Sí, n pequeño, poda básica',    '→ Backtracking (NB04)'),
        ('Sí, con cotas de calidad',      '→ Branch & Bound (NB05)'),
        ('No, solución aproximada sirve', '→ Greedy (NB02) o Aproximación'),
        ('No, espacio muy grande',        '→ Metaheurísticas (simul. annealing)'),
    ]),
]

out_guia = Output()
estado = {'paso': 0}
btn_si  = widgets.Button(description='Sí ✓', button_style='success')
btn_no  = widgets.Button(description='No ✗', button_style='danger')
btn_rst = widgets.Button(description='Reiniciar', button_style='warning')
lbl_q   = widgets.HTML(value='')

def _mostrar_paso():
    paso = estado['paso']
    if paso < len(GUIA):
        lbl_q.value = (f'<p style="font-size:14px;color:{TEXTO};padding:8px;'
                       f'background:white;border-radius:6px;">'
                       f'<b>Pregunta {paso+1}:</b> {GUIA[paso]["pregunta"]}</p>')
        btn_si.disabled  = False
        btn_no.disabled  = False
    else:
        lbl_q.value = ('<p style="font-size:14px;color:#1565C0;padding:8px;'
                       'background:white;border-radius:6px;">'
                       '¡Fin del árbol de decisión! Usa el resumen de arriba '
                       'para elegir el paradigma adecuado.</p>')
        btn_si.disabled = True
        btn_no.disabled = True


def _on_si(b):
    with out_guia:
        out_guia.clear_output()
        paso = estado['paso']
        q    = GUIA[paso]
        if 'si_idx' in q:
            estado['paso'] = q['si_idx']
            _mostrar_paso()
        else:
            print(f'\n{q["si_texto"]}')
            btn_si.disabled = True
            btn_no.disabled = True


def _on_no(b):
    with out_guia:
        out_guia.clear_output()
        paso = estado['paso']
        q    = GUIA[paso]
        print(f'\n{q["no_texto"]}')
        btn_si.disabled = True
        btn_no.disabled = True


def _on_reset(b):
    with out_guia:
        out_guia.clear_output()
    estado['paso'] = 0
    _mostrar_paso()


btn_si.on_click(_on_si)
btn_no.on_click(_on_no)
btn_rst.on_click(_on_reset)

_mostrar_paso()
display(VBox([
    widgets.HTML('<h3 style="color:#212121">Árbol de decisión: ¿qué paradigma uso?</h3>'),
    lbl_q,
    HBox([btn_si, btn_no, btn_rst]),
    out_guia
]))

---
## Autoevaluación final del bloque S03

In [ ]:
# ── Quiz final — Paradigmas Avanzados y repaso S03 ──────────────────────────
import ipywidgets as wg

PREGUNTAS_F = [
    {
        'enunciado': '1. Un algoritmo 2-aproximado para Vertex Cover garantiza que:',
        'opciones': [
            'La solución tiene exactamente 2 nodos',
            'La solución tiene a lo más el doble de nodos que el óptimo',
            'Tarda el doble que el óptimo',
            'Siempre encuentra el óptimo en tiempo polinomial',
        ],
        'correcta': 1,
        'exp': 'La razón de aproximación α=2 significa |ALG| ≤ 2 × |OPT|, no que tenga exactamente 2 nodos.',
    },
    {
        'enunciado': '2. Simulated Annealing puede escapar de óptimos locales porque:',
        'opciones': [
            'Siempre acepta la mejor solución vecina',
            'Acepta soluciones peores con probabilidad exp(Δ/T)',
            'Explora todos los vecinos en paralelo',
            'Usa backtracking para retroceder',
        ],
        'correcta': 1,
        'exp': 'La aceptación probabilística de soluciones peores (con probabilidad que decrece con la temperatura) es la clave que permite escapar de mínimos locales.',
    },
    {
        'enunciado': '3. QuickSort aleatorizado es un algoritmo Las Vegas porque:',
        'opciones': [
            'A veces da una lista incorrectamente ordenada',
            'El resultado es siempre correcto pero el tiempo es aleatorio',
            'El tiempo siempre es O(n log n)',
            'Solo funciona con entradas aleatorias',
        ],
        'correcta': 1,
        'exp': 'Las Vegas: resultado SIEMPRE correcto, tiempo aleatorio. El pivot aleatorio hace que el tiempo esperado sea O(n log n) en cualquier entrada, pero sigue siendo siempre posible ordenar la lista correctamente.',
    },
    {
        'enunciado': '4. El método de Monte Carlo para estimar π tiene un error que crece como:',
        'opciones': [
            'O(1/n) — si doblas los puntos, el error se divide por 2',
            'O(1/√n) — necesitas 4× los puntos para dividir el error por 2',
            'O(1/n²) — converge muy rápido',
            'O(log n) — converge muy lento',
        ],
        'correcta': 1,
        'exp': 'El error del método de Monte Carlo decrece como O(1/√n). Para reducir el error a la mitad, necesitas 4 veces más puntos. Converge más lento que métodos numéricos, pero funciona en dimensiones muy altas.',
    },
    {
        'enunciado': '5. ¿Cuál de los 5 paradigmas del curso (NB01–NB05) es el único que NO garantiza el óptimo en general?',
        'opciones': [
            'Divide y Vencerás',
            'Greedy',
            'Programación Dinámica',
            'Backtracking',
        ],
        'correcta': 1,
        'exp': 'Greedy es el único que no garantiza el óptimo en general. Solo lo hace cuando el problema tiene la propiedad greedy. DP, Backtracking y B&B siempre dan el óptimo (aunque con diferentes costos computacionales).',
    },
]

_sel_f = {}
_out_f = Output()
ctrls  = []
for qi, q in enumerate(PREGUNTAS_F):
    rb = wg.RadioButtons(options=q['opciones'], description='',
                         layout=wg.Layout(width='100%'))
    _sel_f[qi] = rb
    ctrls.append(wg.HTML(f'<b>{q["enunciado"]}</b>'))
    ctrls.append(rb)
    ctrls.append(wg.HTML('<hr style="margin:4px 0"/>'))

btn_f = wg.Button(description='Evaluar (cierre S03)',
                  button_style='primary', icon='graduation-cap')

def _eval_f(b):
    with _out_f:
        _out_f.clear_output()
        score = 0
        for qi, q in enumerate(PREGUNTAS_F):
            idx = q['opciones'].index(_sel_f[qi].value)
            if idx == q['correcta']:
                score += 1
                print(f'✅ {qi+1}. Correcto!')
            else:
                print(f'❌ {qi+1}. Incorrecto. → "{q["opciones"][q["correcta"]]}')
            print(f'   {q["exp"]}')
        pct = score / len(PREGUNTAS_F) * 100
        print(f'\n── Puntaje final: {score}/{len(PREGUNTAS_F)} ({pct:.0f}%) ──')
        if pct == 100:
            print('🎓 ¡Excelente! Dominas los paradigmas avanzados y el bloque S03.')
        elif pct >= 60:
            print('Bien. Repasa las secciones donde fallaste antes del examen.')
        else:
            print('Repasa los notebooks NB02 (Greedy) y NB06 (paradigmas avanzados).')

btn_f.on_click(_eval_f)
display(VBox(ctrls + [btn_f, _out_f]))

---
## Lecturas recomendadas

### Algoritmos de Aproximación

| Recurso | Capítulo |
|---|---|
| **CLRS** — *Introduction to Algorithms* (4ª ed.) | Cap. 35 — Approximation Algorithms |
| **Vazirani** — *Approximation Algorithms* (Springer) | Cap. 1–3 (gratuito en muchas bibliotecas) |

### Metaheurísticas

| Recurso | Contenido |
|---|---|
| **Kirkpatrick et al. (1983)** — *Science* | Artículo original de Simulated Annealing |
| **Russell & Norvig** — *Artificial Intelligence* (4ª ed.) | Cap. 4 — Local Search Algorithms |

### Algoritmos Aleatorizados

| Recurso | Capítulo |
|---|---|
| **Mitzenmacher & Upfal** — *Probability and Computing* (2ª ed.) | Cap. 1–3 (excelente introducción) |
| **Motwani & Raghavan** — *Randomized Algorithms* | Cap. 1 |

### Recursos online

- **VisuAlgo** — [visualgo.net/en](https://visualgo.net/en): visualizaciones interactivas
- **CP-Algorithms** — [cp-algorithms.com](https://cp-algorithms.com): algoritmos con código
- **USACO Guide** — [usaco.guide](https://usaco.guide): currículo estructurado para competitive programming
- **AtCoder Educational DP Contest** — [atcoder.jp/contests/dp](https://atcoder.jp/contests/dp): 26 problemas DP graduados

---
## Cierre del Bloque S03

```
S03 — Diseño de Algoritmos
├── NB00: Introducción y panorama comparativo
├── NB01: Divide y Vencerás     → findMax, Binary Search, Merge Sort
├── NB02: Greedy                → Mochila*, Huffman, Dijkstra
├── NB03: Programación Dinámica → Mochila* (óptimo), Fibonacci, Levenshtein
├── NB04: Backtracking          → Mochila* (árbol completo), N-Reinas
├── NB05: Branch & Bound        → Mochila* (árbol podado), 8-Puzzle
└── NB06: Más allá              → Aproximación, Metaheurísticas, Aleatorizados
         ↑
         * mismos 5 objetos del PDF en los 4 notebooks → comparación directa
```

> **Los 5 objetos del PDF recorrieron 4 paradigmas y demostraron en la práctica por qué no hay un solo algoritmo para todo: cada paradigma brilla en su contexto.**